## Notebook 1/3 — Reddit: Load → Explore → Label → Clean → Balance → Embed

This notebook produces the core artifacts under `outputs/`:
- `reddit_embeddings.npy`
- `reddit_labels.npy`
- `reddit_cleaned_balanced.csv` (for traceability)

Run this first.

In [ ]:
# Colab (optional)
!pip -q install -U pandas numpy scikit-learn sentence-transformers matplotlib seaborn

from pathlib import Path
import numpy as np
import pandas as pd

from pipeline_utils import (
    RANDOM_STATE,
    load_csvs,
    pick_text_column,
    apply_risk_labels,
    clean_text,
    word_count,
    dedupe_by_author_most_recent,
    balance_undersample,
    generate_embeddings,
)

OUTPUT_DIR = Path("outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Outputs:", OUTPUT_DIR.resolve())


[notice] A new release of pip is available: 25.0.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


Outputs: C:\Users\HP\Desktop\major2\outputs


### Step 1 — Load & explore

Set `DATA_PATH` to either:
- a single CSV file, or
- a directory containing many CSVs (loaded recursively).

In [ ]:
def explore_dataframe(df: pd.DataFrame, subreddit_col: str = "subreddit", n: int = 5) -> None:
    print("Shape:", df.shape)
    print("\nColumns:")
    print(list(df.columns))
    print("\nSample rows:")
    display(df.head(n))

    if subreddit_col in df.columns:
        print("\nSubreddit value counts (top 30):")
        display(df[subreddit_col].value_counts(dropna=False).head(30))

    print("\nNull counts (top 50):")
    display(df.isna().sum().sort_values(ascending=False).head(50))


CANDIDATE_DATA_PATHS = [
    r"Original Reddit Data/raw data",
    "/content/reddit-mental-health-dataset",
    "/content/reddit_dataset",
    "/content/dataset",
    "/content/drive/MyDrive/reddit-mental-health-dataset",
]
DATA_PATH = next((p for p in CANDIDATE_DATA_PATHS if Path(p).exists()), None)
if DATA_PATH is None:
    raise FileNotFoundError(
        "Could not auto-detect the Reddit dataset location. Set DATA_PATH to your CSV file or a directory of CSVs, then re-run."
    )

print("Using DATA_PATH:", DATA_PATH)
reddit_df_raw, files_loaded = load_csvs(DATA_PATH)
print(f"Loaded {len(files_loaded)} file(s).")
explore_dataframe(reddit_df_raw)

Using DATA_PATH: Original Reddit Data/raw data
Loaded 219 file(s).
Shape: (1851580, 8)

Columns:
['Unnamed: 0', 'author', 'created_utc', 'score', 'selftext', 'subreddit', 'title', 'timestamp']

Sample rows:


,Unnamed: 0,author,created_utc,score,selftext,subreddit,title,timestamp
0,0,erbush1988,1556632225,4,Hello all. \n\nMy wife has anxiety and lately...,Anxiety,Wife has anxiety. How can I help?,2019-04-30 23:50:25
1,1,weeblybeebly,1556631109,4,\n I wanted to write this because I...,Anxiety,My Anxiety’s Kryptonite.,2019-04-30 23:31:49
2,2,logicminds,1556630422,1,NaN,Anxiety,Do you guys ever make friends online/apps to a...,2019-04-30 23:20:22
3,3,kweesnaw,1556629580,2,"Hi all, so I've been taking Effexor XR 75 MG f...",Anxiety,"While taking Effexor, is it okay to take Cloni...",2019-04-30 23:06:20
4,4,drekiaa,1556628567,8,Hi guys!\n\nI've finally come to the conclusio...,Anxiety,"After Accepting You Need Help, What Was the Fi...",2019-04-30 22:49:27



Subreddit value counts (top 30):


subreddit
depression                                                                                                                                                                                                                             624561
SuicideWatch                                                                                                                                                                                                                           483048
mentalhealth                                                                                                                                                                                                                           303109
Anxiety                                                                                                                                                                                                                                280038
lonely                                


Null counts (top 50):


selftext       54564
title              8
Unnamed: 0         0
author             0
score              0
created_utc        0
subreddit          0
timestamp          0
dtype: int64

### Step 2 — Label engineering

Applies the 3-class subreddit mapping and drops unmapped subreddits.

In [ ]:
reddit_df_labeled = apply_risk_labels(reddit_df_raw)
print("After labeling + dropping unmapped subreddits:")
print("Shape:", reddit_df_labeled.shape)
print("\nClass distribution:")
display(reddit_df_labeled["risk_label_name"].value_counts())

present = set(reddit_df_labeled["risk_label"].unique().tolist())
missing = {0, 1} - present
if missing:
    raise ValueError(
        "After applying the required subreddit-to-label mapping, "
        f"the dataset is missing label(s): {sorted(missing)}.\n"
        f"Present labels: {sorted(present)}.\n"
        "This usually means your CSVs don't contain both suicidewatch (High Risk) "
        "and at least one of: depression, anxiety, lonely, mentalhealth (Mental Health Risk).\n"
        "Fix: update DATA_PATH to include those subreddits' CSV files, then re-run from Step 1."
    )

After labeling + dropping unmapped subreddits:
Shape: (1847871, 10)

Class distribution:


risk_label_name
Mental Health Risk      1364823
High Risk (Suicidal)     483048
Name: count, dtype: int64

### Step 3 — Cleaning + leakage controls + balancing

- Cleans `selftext`/`body`
- Drops `[deleted]`/`[removed]`/empty
- Drops posts with < 20 words
- Deduplicates by `author` (keeps most recent)
- Undersamples to **2000 per class** (or max possible)

Saves `outputs/reddit_cleaned_balanced.csv` for traceability.

In [ ]:
TEXT_COL = pick_text_column(reddit_df_labeled)
print("Using text column:", TEXT_COL)

reddit_df = reddit_df_labeled.copy()
reddit_df["text_raw"] = reddit_df[TEXT_COL]
reddit_df["text_clean"] = reddit_df["text_raw"].apply(clean_text)
reddit_df = reddit_df[reddit_df["text_clean"].astype(bool)].copy()

reddit_df["word_count"] = reddit_df["text_clean"].apply(word_count)
reddit_df = reddit_df[reddit_df["word_count"] >= 20].copy()

reddit_df = dedupe_by_author_most_recent(reddit_df)

class_counts_before = (
    reddit_df["risk_label_name"]
    .value_counts()
    .reindex(["Control", "Mental Health Risk", "High Risk"], fill_value=0)
)
print("Class distribution (after cleaning + dedup, before balancing):")
display(class_counts_before)
print("Dataset size:", len(reddit_df))

reddit_df_balanced, effective_target = balance_undersample(reddit_df, target_per_class=2000)
class_counts_after = (
    reddit_df_balanced["risk_label_name"]
    .value_counts()
    .reindex(["Control", "Mental Health Risk", "High Risk"], fill_value=0)
)

print(f"\nBalanced to {effective_target} per class.")
print("Final class distribution (after balancing):")
display(class_counts_after)
print("Final dataset size:", len(reddit_df_balanced))

clean_csv_path = OUTPUT_DIR / "reddit_cleaned_balanced.csv"
reddit_df_balanced[[
    "risk_label",
    "risk_label_name",
    "subreddit",
    "author" if "author" in reddit_df_balanced.columns else None,
    "created_utc" if "created_utc" in reddit_df_balanced.columns else None,
    "timestamp" if "timestamp" in reddit_df_balanced.columns else None,
    "word_count",
    "text_clean",
]].loc[:, lambda d: [c for c in d.columns if c is not None]].to_csv(clean_csv_path, index=False)

print("Saved:", clean_csv_path.resolve())

Using text column: selftext
Class distribution (after cleaning + dedup, before balancing):


risk_label_name
Control                    0
Mental Health Risk    572014
High Risk                  0
Name: count, dtype: int64

Dataset size: 752155

Balanced to 2000 per class.
Final class distribution (after balancing):


c:\Users\HP\Desktop\major2\pipeline_utils.py:175: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(lambda g: g.sample(n=effective_target, random_state=RANDOM_STATE, replace=False))


risk_label_name
Control                  0
Mental Health Risk    2000
High Risk                0
Name: count, dtype: int64

Final dataset size: 4000
Saved: C:\Users\HP\Desktop\major2\outputs\reddit_cleaned_balanced.csv


### Step 4 — MiniLM embeddings

Generates one embedding per post using `all-MiniLM-L6-v2` and saves:
- `outputs/reddit_embeddings.npy`
- `outputs/reddit_labels.npy`

In [ ]:
#pip uninstall tensorflow tensorflow-cpu tensorflow-gpu keras -y
!pip install "numpy<2" --force-reinstall
!pip install sentence-transformers --force-reinstall

  Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl.metadata (61 kB)
Using cached numpy-1.26.4-cp310-cp310-win_amd64.whl (15.8 MB)
  Attempting uninstall: numpy
    Found existing installation: numpy 2.2.6
    Uninstalling numpy-2.2.6:
      Successfully uninstalled numpy-2.2.6


  You can safely remove it manually.
  You can safely remove it manually.
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
colormap 1.2.0 requires numpy<3,>=2; python_version >= "3.9" and python_version < "4.0", but you have numpy 1.26.4 which is incompatible.
streamlit 1.40.1 requires packaging<25,>=20, but you have packaging 26.0 which is incompatible.
streamlit 1.40.1 requires rich<14,>=10.14.0, but you have rich 14.3.3 which is incompatible.
tensorflow-intel 2.18.0 requires keras>=3.5.0, but you have keras 2.8.0 which is incompatible.
tensorflow-intel 2.18.0 requires ml-dtypes<0.5.0,>=0.4.0, but you have ml-dtypes 0.5.0 which is incompatible.
tensorflow-intel 2.18.0 requires tensorboard<2.19,>=2.18, but you have tensorboard 2.8.0 which is incompatible.
torchvision 0.21.0 requires torch==2.6.0, but you have torch 2.10.0 which is incompatible.

[notice] A new 

^C


In [ ]:
from typing import Dict, List, Optional, Tuple, Iterable
def generate_embeddings(
    texts: List[str],
    model_name: str = "all-MiniLM-L6-v2",
    batch_size: int = 64,
    normalize_embeddings: bool = False,
) -> np.ndarray:
    import os
    os.environ["USE_TF"] = "0"
    os.environ["TRANSFORMERS_NO_TF"] = "1"
    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

    import importlib, sys
    # Block tensorflow from being imported by sentence_transformers
    sys.modules["tensorflow"] = None

    from sentence_transformers import SentenceTransformer
    import torch

    device = "cuda" if torch.cuda.is_available() else "cpu"

    model = SentenceTransformer(model_name, device=device)
    emb = model.encode(
        texts,
        batch_size=batch_size,
        show_progress_bar=True,
        convert_to_numpy=True,
        normalize_embeddings=normalize_embeddings,
    )
    return emb.astype(np.float32)

In [ ]:
import os
os.environ["USE_TF"] = "0"
os.environ["TRANSFORMERS_NO_TF"] = "1"
os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"
import sys
sys.modules["tensorflow"] = None

import numpy as np
import pandas as pd
from pathlib import Path
from sentence_transformers import SentenceTransformer

OUTPUT_DIR = Path("outputs")

# Load from already-saved CSV
cleaned = pd.read_csv(OUTPUT_DIR / "reddit_cleaned_balanced.csv")
texts = cleaned["text_clean"].tolist()
labels = cleaned["risk_label"].to_numpy()

print(f"Loaded {len(texts)} texts from CSV")
print("Label distribution:", dict(zip(*np.unique(labels, return_counts=True))))

# Generate embeddings
model = SentenceTransformer("all-MiniLM-L6-v2", device="cpu")
embeddings = model.encode(
    texts,
    batch_size=64,
    show_progress_bar=True,
    convert_to_numpy=True,
)
embeddings = embeddings.astype(np.float32)

# Save
emb_path = OUTPUT_DIR / "reddit_embeddings.npy"
lab_path = OUTPUT_DIR / "reddit_labels.npy"
np.save(emb_path, embeddings)
np.save(lab_path, labels)

print("Embedding matrix shape:", embeddings.shape)
print("Labels shape:", labels.shape)
print("Saved:", emb_path.resolve())
print("Saved:", lab_path.resolve())

OSError: [WinError 1114] A dynamic link library (DLL) initialization routine failed. Error loading "c:\Users\HP\AppData\Local\Programs\Python\Python310\lib\site-packages\torch\lib\c10.dll" or one of its dependencies.